# 04 — Game Similarity Graph

We build a graph where each node is a game and each edge connects a game to one of its 10 most similar games. The edge weight is the Jaccard similarity.

In this notebook, we build a graph where each node is a game and each edge connects a game to one of its 10 most similar games. The edge weight is the Jaccard similarity.

In [14]:
from pathlib import Path

import networkx as nx
import pandas as pd


candidates = pd.read_parquet("../data/processed/game_similarity_candidates.parquet")

# Count each pair in both directions, so every game sees all its neighbours.
both_directions = pd.concat([
    candidates,
    candidates.rename(columns={"game_a": "game_b", "game_b": "game_a"})
])

edges = (
    both_directions
    .sort_values("jaccard_similarity", ascending=False)
    .groupby("game_a")
    .head(10)
)

G = nx.from_pandas_edgelist(edges, "game_a", "game_b", edge_attr="jaccard_similarity")

print("Games:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Games: 2363
Edges: 18073


In [15]:
OUTPUT_PATH = Path("../data/processed/game_graph_edges.parquet")

nx.to_pandas_edgelist(G, source="game_a", target="game_b").to_parquet(OUTPUT_PATH, index=False)

print("Saved graph to:", OUTPUT_PATH)

Saved graph to: ..\data\processed\game_graph_edges.parquet


## Graph diagnostics

Before applying community detection, we inspect a few basic properties of the graph to make sure the construction behaves as expected.

In [16]:
# Basic graph diagnostics.
all_games = set(candidates["game_a"]) | set(candidates["game_b"])
graph_games = set(G.nodes())

missing_games = all_games - graph_games

degrees = [degree for _, degree in G.degree()]
edge_weights = [
    data["jaccard_similarity"]
    for _, _, data in G.edges(data=True)
]

print("Games in candidate data:", len(all_games))
print("Games in graph:", G.number_of_nodes())
print("Games excluded from graph:", len(missing_games))

print("\nDegree:")
print("Minimum:", min(degrees))
print("Median:", pd.Series(degrees).median())
print("Maximum:", max(degrees))

print("\nEdge similarity:")
print("Minimum:", min(edge_weights))
print("Median:", pd.Series(edge_weights).median())
print("Maximum:", max(edge_weights))

print("\nConnected components:", nx.number_connected_components(G))
print(
    "Largest component:",
    len(max(nx.connected_components(G), key=len)),
    "games"
)

Games in candidate data: 2363
Games in graph: 2363
Games excluded from graph: 0

Degree:
Minimum: 1
Median: 13.0
Maximum: 105

Edge similarity:
Minimum: 0.001768867924528302
Median: 0.071697022260769
Maximum: 0.8939393939393939

Connected components: 2
Largest component: 2361 games


### Graph diagnostics

The graph contains all 2,363 games that appeared in the LSH candidate pairs, with 18,074 weighted edges. The median degree is 13, meaning that most games are connected to a relatively small number of similar games.

Although some edges have low Jaccard similarity, the edge weights preserve the strength of each relationship. The graph is also almost fully connected: 2,361 of the 2,363 games belong to the largest connected component.

Overall, the top-10 neighbour construction provides broad coverage while keeping the graph relatively sparse, making it suitable for the following community detection analysis.

## Louvain community detection

We now apply the Louvain algorithm to identify communities of games with densely connected player audiences.

The algorithm uses the Jaccard similarity as the edge weight, so stronger behavioural similarities contribute more to the detected community structure.

In [17]:
communities = nx.community.louvain_communities(
    G,
    weight="jaccard_similarity",
    seed=42
)

modularity = nx.community.modularity(
    G,
    communities,
    weight="jaccard_similarity"
)

print("Number of communities:", len(communities))
print("Modularity:", modularity)

community_sizes = sorted(
    [len(community) for community in communities],
    reverse=True
)

print("Largest community sizes:", community_sizes[:10])

Number of communities: 15
Modularity: 0.6835558284685391
Largest community sizes: [679, 641, 414, 117, 114, 112, 74, 62, 55, 54]


## Inspect detected communities

To interpret the communities, we attach game names to each node and inspect representative games from each detected group.

In [19]:
# Map each game to its Louvain community.
game_to_community = {}

for community_id, community in enumerate(communities):
    for game_id in community:
        game_to_community[game_id] = community_id

interactions = pd.read_parquet(
    "../data/processed/interactions_filtered.parquet"
)

# Get game names from the interaction data.
game_names = (
    interactions[["game_id", "game_name"]]
    .drop_duplicates(subset=["game_id"])
)


community_df = pd.DataFrame(
    list(game_to_community.items()),
    columns=["game_id", "community"]
)

community_df = community_df.merge(
    game_names,
    on="game_id",
    how="left"
)

community_df.head()

,game_id,community,game_name
0,8210,0,Sam & Max 102: Situation: Comedy
1,98600,0,"Demolition, Inc."
2,31180,0,Tales of Monkey Island: Chapter 2 - The Siege ...
3,94510,0,Back to the Future: Ep 3 - Citizen Brown
4,31240,0,Sam & Max 303: They Stole Max's Brain!


### Representative games in each community

To better understand the detected communities, we inspect the most strongly connected games within each group. Games with a high weighted degree have strong similarity relationships with several other games in the graph and can therefore serve as useful representatives of their community.

In [21]:
# Calculate each game's weighted degree.
weighted_degree = dict(
    G.degree(weight="jaccard_similarity")
)

community_df["weighted_degree"] = (
    community_df["game_id"]
    .map(weighted_degree)
)

# Select up to five strongly connected games from each community.
representative_games = (
    community_df
    .sort_values(
        ["community", "weighted_degree"],
        ascending=[True, False]
    )
    .groupby("community")
    .head(5)
)

# Print representative games community by community.
for community_id in sorted(representative_games["community"].unique()):
    games = representative_games[
        representative_games["community"] == community_id
    ]

    print(f"\nCommunity {community_id}")
    print("-" * 40)

    for _, row in games.iterrows():
        print(
            f"{row['game_name']} "
            f"(weighted degree: {row['weighted_degree']:.2f})"
        )


Community 0
----------------------------------------
Sam & Max 204: Chariots of the Dogs (weighted degree: 5.98)
Sam & Max 205: What's New Beelzebub? (weighted degree: 5.83)
Sam & Max 201: Ice Station Santa (weighted degree: 5.81)
Sam & Max 203: Night of the Raving Dead (weighted degree: 5.75)
Sam & Max 304: Beyond the Alley of the Dolls (weighted degree: 5.60)

Community 1
----------------------------------------
Call of Duty: Black Ops (weighted degree: 2.88)
Call of Duty: Black Ops II - Multiplayer (weighted degree: 2.80)
Call of Duty: Black Ops - Multiplayer (weighted degree: 2.58)
Call of Duty: Black Ops II - Zombies (weighted degree: 2.43)
Call of Duty: Modern Warfare 3 (weighted degree: 2.41)

Community 2
----------------------------------------
Call of Duty: Black Ops - OSX (weighted degree: 0.58)
Call of Duty: Black Ops - Multiplayer OSX (weighted degree: 0.58)

Community 3
----------------------------------------
Blood of Old (weighted degree: 9.88)
SUPER DISTRO (weighted de

### Initial community interpretation

Several communities show clear behavioural structure. For example, separate communities emerge around *Sam & Max*, *Call of Duty*, strategy games such as *Total War* and *Crusader Kings*, major mainstream Steam titles, and SEGA/Sonic games.

Other communities are less obvious from game titles alone. This is expected because the communities are based entirely on shared player behaviour rather than genre or metadata. These less obvious groups may therefore capture relationships such as shared audiences, bundles, franchises, or similar purchasing and playing patterns.

## Community interpretation using game genres

The Louvain communities were created only from shared player behaviour. We now use Steam genre metadata afterward to help interpret whether the detected communities also show meaningful thematic structure.

In [22]:
# Load game metadata.
games = pd.read_parquet(
    "../data/processed/games_clean.parquet"
)

# Attach genres to the detected communities.
community_genres = community_df.merge(
    games[["game_id", "genres"]],
    on="game_id",
    how="left"
)

print(
    "Games with genre metadata:",
    community_genres["genres"].notna().sum()
)

print(
    "Games without genre metadata:",
    community_genres["genres"].isna().sum()
)

Games with genre metadata: 2001
Games without genre metadata: 362


### Dominant genres within communities

We count the most common Steam genres within each detected community. This provides an external interpretation of the behavioural communities, since genre information was not used during graph construction or community detection.

In [23]:
# Expand the genre lists so each genre becomes its own row.
genre_rows = (
    community_genres
    .dropna(subset=["genres"])
    .explode("genres")
)

# Count genres within each community.
genre_counts = (
    genre_rows
    .groupby(["community", "genres"])
    .size()
    .reset_index(name="count")
)

# Show the five most common genres in each community.
top_genres = (
    genre_counts
    .sort_values(
        ["community", "count"],
        ascending=[True, False]
    )
    .groupby("community")
    .head(5)
)

for community_id in sorted(top_genres["community"].unique()):
    genres = top_genres[
        top_genres["community"] == community_id
    ]

    print(f"\nCommunity {community_id}")
    print("-" * 30)

    for _, row in genres.iterrows():
        print(f"{row['genres']}: {row['count']}")


Community 0
------------------------------
Adventure: 23
Action: 11
RPG: 7
Casual: 6
Indie: 5

Community 1
------------------------------
Action: 6

Community 2
------------------------------
Action: 1

Community 3
------------------------------
Indie: 56
Action: 51
Adventure: 28
Casual: 22
Strategy: 20

Community 4
------------------------------
Indie: 71
Action: 40
Adventure: 37
Casual: 31
Strategy: 19

Community 5
------------------------------
Indie: 93
Adventure: 57
Action: 47
Casual: 40
RPG: 29

Community 7
------------------------------
Indie: 413
Action: 324
Adventure: 223
RPG: 153
Strategy: 130

Community 8
------------------------------
Action: 29
Indie: 22
RPG: 21
Casual: 10
Strategy: 10

Community 9
------------------------------
Strategy: 203
Action: 125
Indie: 108
RPG: 98
Simulation: 94

Community 10
------------------------------
Action: 426
Indie: 243
Adventure: 200
RPG: 123
Free to Play: 106

Community 11
------------------------------
Action: 11
Adventure: 8
Casual: 

### Genre proportions within communities

Because the detected communities differ greatly in size, raw genre counts are not directly comparable. We therefore also calculate the proportion of games within each community associated with each genre.

In [24]:
# Count games with genre metadata in each community.
community_sizes_with_genres = (
    community_genres
    .dropna(subset=["genres"])
    .groupby("community")["game_id"]
    .nunique()
)

genre_counts["percentage"] = genre_counts.apply(
    lambda row: (
        row["count"]
        / community_sizes_with_genres[row["community"]]
        * 100
    ),
    axis=1
)

top_genre_percentages = (
    genre_counts
    .sort_values(
        ["community", "percentage"],
        ascending=[True, False]
    )
    .groupby("community")
    .head(5)
)

for community_id in sorted(
    top_genre_percentages["community"].unique()
):
    genres = top_genre_percentages[
        top_genre_percentages["community"] == community_id
    ]

    print(f"\nCommunity {community_id}")
    print("-" * 30)

    for _, row in genres.iterrows():
        print(
            f"{row['genres']}: "
            f"{row['percentage']:.1f}%"
        )


Community 0
------------------------------
Adventure: 65.7%
Action: 31.4%
RPG: 20.0%
Casual: 17.1%
Indie: 14.3%

Community 1
------------------------------
Action: 100.0%

Community 2
------------------------------
Action: 100.0%

Community 3
------------------------------
Indie: 66.7%
Action: 60.7%
Adventure: 33.3%
Casual: 26.2%
Strategy: 23.8%

Community 4
------------------------------
Indie: 81.6%
Action: 46.0%
Adventure: 42.5%
Casual: 35.6%
Strategy: 21.8%

Community 5
------------------------------
Indie: 86.9%
Adventure: 53.3%
Action: 43.9%
Casual: 37.4%
RPG: 27.1%

Community 7
------------------------------
Indie: 69.9%
Action: 54.8%
Adventure: 37.7%
RPG: 25.9%
Strategy: 22.0%

Community 8
------------------------------
Action: 47.5%
Indie: 36.1%
RPG: 34.4%
Casual: 16.4%
Strategy: 16.4%

Community 9
------------------------------
Strategy: 59.2%
Action: 36.4%
Indie: 31.5%
RPG: 28.6%
Simulation: 27.4%

Community 10
------------------------------
Action: 73.6%
Indie: 42.0%
Adven

### Genre interpretation

The genre distributions support several of the behavioural communities found by Louvain. For example, Community 0 is strongly associated with Adventure games, Community 9 with Strategy games, Community 10 with Action games, and Community 12 with Simulation, Racing, and Sports.

Other communities have more mixed genre distributions, suggesting that shared player behaviour captures relationships that are not explained by genre alone. This is expected because genre information was not used when constructing the graph or detecting communities.